In [ ]:
"""
✨ 튜터링 스크립트: 한국어 뉴스 토픽 분류 분석 (Korean News Topic Classification) ✨

안녕하세요! 👋 저는 당신의 친절하고 위트 넘치는 AI 코딩 튜터입니다.
이 데이터셋은 한국어 뉴스 헤드라인을 보고 '이 뉴스가 어떤 분야에 속할까?'를 분류하는 아주 재미있는 학습 자료예요.

[데이터셋 정보]
- 데이터셋명: hugmanskj/korean-news-topic-classification
- 언어: 한국어 (ko)
- 태스크: 텍스트 분류 (Text Classification)
- 목표: 주어진 뉴스 기사 제목(text)이 경제, 사회, 생활문화, IT과학 중 어디에 속하는지 예측해보기!

🚨 주의 사항: 이 실습은 데이터셋의 특성을 파악하는 '탐색'에 초점을 맞춥니다.
실제 AI 모델을 훈련시키려면 토크나이저와 모델 로딩이 필요해요! (다음 단계에서 배울 거예요 😉)
"""

import random
import pandas as pd
from datasets import load_dataset
import warnings

# 경고 메시지 무시 (튜터링 코드를 깔끔하게 만들기 위함)
warnings.filterwarnings("ignore")

# ====================================================================
# ⚙️ 1. 환경 설정 및 데이터 로드
# ====================================================================

DATASET_NAME = "hugmanskj/korean-news-topic-classification"
SAMPLE_COUNT = 100  # 실습의 효율을 위해 상위 100개 샘플만 사용합니다.

print("🌟 [튜터링 시작] 데이터셋 로딩을 시작합니다...")

# 📌 데이터 로드 전략: 스트리밍을 먼저 시도하고, 실패하면 전체를 로드합니다.
dataset = None
try:
    # 1. 스트리밍 모드 (가장 빠르고 메모리 효율적)를 시도합니다.
    print("➡️ Attempting to load data in streaming mode (한 번에 전체 로딩 시도)...")
    # 스트리밍은 보통 전체 split에 대해 시도합니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 데이터셋을 로드했습니다. (메모리 절약 모드 🚀)")

except Exception as e:
    # 2. 스트리밍 실패 시 (혹은 환경 제약 시), 작은 test split만 일반 모드로 로드하여 진행합니다.
    print(f"⚠️ 스트리밍 로드 중 오류가 발생했습니다. ({e})")
    print("➡️ 테스트 데이터셋을 일반 모드로 로드하여 진행합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='test')
    except Exception as e_alt:
        print(f"🚨 죄송해요! 데이터셋 로드에 실패했습니다. {e_alt}")
        exit()


# 📌 스트리밍/일반 데이터셋 공통 처리: 샘플링 루틴
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)인 경우
    print(f"\n🔎 상위 {SAMPLE_COUNT}개의 샘플만 추출합니다. (스트리밍 모드)")
    # Sample Iterator를 생성합니다.
    sample_iterator = iter(dataset.take(SAMPLE_COUNT))
    # 메모리 효율을 위해 샘플을 리스트로 변환하는 대신, 반복자(iterator)로 사용합니다.
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    print(f"\n🔎 상위 {SAMPLE_COUNT}개의 샘플만 추출합니다. (일반 모드)")
    # 전체를 리스트로 로드합니다.
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))


# ====================================================================
# 🧠 2. 데이터 전역 분석 (EDA: 탐색적 데이터 분석)
# ====================================================================

print("\n" + "="*80)
print("🔬 [Step 1] 전역 분포 및 키워드 분석: 이 데이터셋은 어떤 주제가 제일 많을까요?")
print("="*80)

# 카테고리 매핑을 쉽게 하기 위한 딕셔너리
LABEL_MAP = {
    0: "💰 경제", 
    1: "🤝 사회", 
    2: "🎨 생활문화", 
    3: "🔬 IT과학"
}

# 📚 전체 샘플의 레이블 분포 분석
label_counts = pd.Series([sample['label_name'] for sample in sample_data_list]).value_counts()

print("\n[📊 카테고리별 출현 빈도]")
for label, count in label_counts.items():
    print(f"  ✨ {label}: {count}회 (가장 많은 주제입니다!)")

# 🔑 주요 키워드 분석 (매우 간단한 규칙 기반 탐색)
def analyze_keywords(data_list):
    """데이터셋 전체에서 특정 키워드가 어떤 주제에 많이 등장하는지 분석합니다."""
    keyword_map = {
        "코스피": "💰 경제",
        "증시": "💰 경제",
        "정부": "🤝 사회",
        "사건": "🤝 사회",
        "맛집": "🎨 생활문화",
        "여행": "🎨 생활문화",
        "인공지능": "🔬 IT과학",
        "AI": "🔬 IT과학"
    }
    
    keyword_topic_score = {key: {v: 0 for v in LABEL_MAP.values()} for key in keyword_map}
    
    for sample in data_list:
        text = sample['text']
        actual_topic = LABEL_MAP[sample['label']]
        
        for keyword, expected_topic in keyword_map.items():
            if keyword in text and actual_topic == expected_topic:
                # 키워드가 존재하고, 그 주제가 예상되는 경우 점수를 높입니다.
                keyword_topic_score[keyword][actual_topic] += 1
            elif keyword in text:
                # 키워드는 있지만 주제가 예상과 다를 경우 (데이터의 복잡성 반영)
                 keyword_topic_score[keyword][actual_topic] += 0.5

    return keyword_topic_score

print("\n[💡 핵심 키워드 출현 분석 (Top 3)]")
keyword_scores = analyze_keywords(sample_data_list)

# 가장 점수가 높은 키워드와 그 이유를 간략하게 보여줍니다.
top_keywords = random.sample(list(keyword_scores.keys()), min(3, len(keyword_scores)))

for keyword in top_keywords:
    scores = keyword_scores[keyword]
    # 가장 높은 점수를 받은 주제를 찾아 출력
    most_likely_topic = max(scores, key=scores.get)
    score = scores[most_likely_topic]
    
    print(f"  🔑 '{keyword}' 키워드는 주로 {most_likely_topic}와 관련이 깊습니다. (추정 점수: {score:.1f})")


# ====================================================================
# 🤖 3. 창의적인 AI 실습 예시: '오늘의 뉴스 헤드라인 분석가'
# ====================================================================

print("\n" + "="*80)
print("🏆 [Step 2] 오늘의 뉴스 헤드라인 분석가 실습: 주제 예측 챌린지!")
print("===================================================================")

def topic_predictor(text):
    """
    주어진 텍스트를 바탕으로 가장 유력한 토픽을 추론하는 간단한 규칙 기반 함수입니다.
    (실제 AI는 훨씬 복잡한 방법을 사용해요!)
    """
    text = text.lower()
    
    # 🚀 경제 관련 키워드 검사
    if any(k in text for k in ["코스피", "주가", "금리", "경제", "물가"]):
        return "💰 경제 (Economy)"
    
    # 🧑‍🤝‍🧑 사회 관련 키워드 검사
    if any(k in text for k in ["사건", "사고", "정부", "정책", "사회"]):
        return "🤝 사회 (Society)"
    
    # 🖼️ 생활문화 관련 키워드 검사
    if any(k in text for k in ["맛집", "여행", "예술", "문화", "패션"]):
        return "🎨 생활문화 (Culture/Life)"
    
    # 🛰️ IT과학 관련 키워드 검사
    if any(k in text for k in ["인공지능", "AI", "반도체", "과학", "지구온난화"]):
        return "🔬 IT과학 (Tech/Science)"
        
    return "❓ 예측 불가 (Unknown)"


print(f"\n🌟 {len(sample_data_list)}개의 샘플 중 무작위로 5개의 헤드라인을 뽑아 분석해 봅시다!")

# 샘플 중 5개를 무작위로 선택합니다.
sample_to_analyze = random.sample(sample_data_list, min(5, len(sample_data_list)))

print("\n-----------------------------------------------------------------")
for i, sample in enumerate(sample_to_analyze):
    headline = sample['text']
    true_label = sample['label_name']
    
    # 🧠 우리가 만든 가상의 AI가 예측하는 주제
    predicted_topic = topic_predictor(headline)
    
    print(f"\n[✨ No.{i+1} 헤드라인] '{headline}'")
    print(f"  💡 우리가 예측한 주제: {predicted_topic}")
    print(f"  ✅ 실제 정답 주제: {true_label}")
    
    if predicted_topic == true_label:
        print("  ✨ 튜터 평가: 와우! 🤖👍 아주 잘 맞추셨어요! (규칙 기반 예측 성공!)")
    else:
        print("  🤔 튜터 평가: 🤔 조금 틀렸지만, 왜 틀렸는지 분석해보는 게 진짜 공부예요!")

print("\n" + "="*80)
print("🎉 수고하셨습니다! ✨")
print("지금까지 우리는 데이터셋의 분포를 살펴보고, 간단한 키워드를 활용해 토픽을 예측하는 과정을 거쳤습니다.")
print("이 원리들을 조금만 발전시키면, 강력한 한국어 뉴스 분류 모델을 만들 수 있게 될 거예요!")